# Lab Manual 4: Scikit-Learn Preprocessing and Modeling Pipeline
**Course:** Machine Learning  
**Topic:** Building a Leakage-Safe Scikit-Learn Preprocessing and Modeling Pipeline  
**Prepared by:** Sharad Laad | ORY AI Labs  
**LinkedIn:** [Sharad Laad](https://www.linkedin.com/in/sharadlaad/)  

---

### Lab Overview & Objectives
The objective of this lab is to build a complete Machine Learning workflow using Scikit-Learn pipelines.
Instead of scattering manual preprocessing steps across many notebook cells, preprocessing and modeling are unified into a single, reusable, leakage-safe pipeline:


In [1]:
# Step 1: Import Required Libraries
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Step 2: Load the Dataset
# Use flexible path detection
possible_paths = [
    Path("../data/processed/olist_orders_abt.csv"),
    Path("data/processed/olist_orders_abt.csv"),
    Path("olist_orders_abt.csv")
]

data_path = None
for path in possible_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError("Could not find olist_orders_abt.csv. Please check the file path.")

df = pd.read_csv(data_path)
df.head()

,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_year,order_month,order_day,order_day_of_week,...,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category,delivery_speed_category,payment_value_band
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017,10,2,0,...,2.0,voucher,1.0,29.99,8.72,1.0,1.0,housewares,Normal,Low
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018,7,24,1,...,1.0,boleto,1.0,118.70,22.76,1.0,1.0,perfumery,Normal,High
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018,8,8,2,...,1.0,credit_card,1.0,159.90,19.22,1.0,1.0,auto,Normal,Premium
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,delivered,2017,11,18,5,...,1.0,credit_card,1.0,45.00,27.20,1.0,1.0,pet_shop,Normal,Medium
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018,2,13,1,...,1.0,credit_card,1.0,19.90,8.72,1.0,1.0,stationery,Fast,Low


In [3]:
# Step 3: Inspect Data
print("Shape:", df.shape)
df.info()
df.describe()

Shape: (99441, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  99441 non-null  object 
 1   customer_id               99441 non-null  object 
 2   customer_unique_id        99441 non-null  object 
 3   customer_city             99441 non-null  object 
 4   customer_state            99441 non-null  object 
 5   order_status              99441 non-null  object 
 6   order_year                99441 non-null  int64  
 7   order_month               99441 non-null  int64  
 8   order_day                 99441 non-null  int64  
 9   order_day_of_week         99441 non-null  int64  
 10  order_hour                99441 non-null  int64  
 11  delivery_days             96476 non-null  float64
 12  estimated_delivery_days   99441 non-null  int64  
 13  delivery_delay_days       96476 non-null  

,order_year,order_month,order_day,order_day_of_week,order_hour,delivery_days,estimated_delivery_days,delivery_delay_days,is_late_delivery,review_score,...,review_comment_count,has_review_comment,total_payment_value,max_payment_installments,payment_types_count,total_items,total_price,total_freight,unique_products,unique_sellers
count,99441.000000,99441.000000,99441.000000,99441.000000,99441.000000,96476.000000,99441.000000,96476.000000,99441.000000,98673.000000,...,98673.000000,98673.000000,99440.000000,99440.000000,99440.000000,98666.000000,98666.000000,98666.000000,98666.000000,98666.000000
mean,2017.539838,6.032220,15.505948,2.755735,14.770829,12.094086,23.403958,-11.876881,0.065717,4.086793,...,0.415281,0.413852,160.990267,2.930521,1.022586,1.141731,137.754076,22.823562,1.038098,1.013622
std,0.505007,3.232999,8.667298,1.966495,5.326800,9.551746,8.829562,10.183854,0.247789,1.346274,...,0.495685,0.492525,221.951257,2.715685,0.148582,0.538452,210.645145,21.650909,0.226456,0.122297
min,2016.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,-147.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.850000,0.000000,1.000000,1.000000
25%,2017.000000,3.000000,8.000000,1.000000,11.000000,6.000000,18.000000,-17.000000,0.000000,4.000000,...,0.000000,0.000000,62.010000,1.000000,1.000000,1.000000,45.900000,13.850000,1.000000,1.000000
50%,2018.000000,6.000000,15.000000,3.000000,15.000000,10.000000,23.000000,-12.000000,0.000000,5.000000,...,0.000000,0.000000,105.290000,2.000000,1.000000,1.000000,86.900000,17.170000,1.000000,1.000000
75%,2018.000000,8.000000,23.000000,4.000000,19.000000,15.000000,28.000000,-7.000000,0.000000,5.000000,...,1.000000,1.000000,176.970000,4.000000,1.000000,1.000000,149.900000,24.040000,1.000000,1.000000
max,2018.000000,12.000000,31.000000,6.000000,23.000000,209.000000,155.000000,188.000000,1.000000,5.000000,...,3.000000,1.000000,13664.080000,24.000000,2.000000,21.000000,13440.000000,1794.960000,8.000000,5.000000


### Student Checkpoint 1
1. **How many rows are there?** 99,441 rows.
2. **How many columns are there?** 31 columns.
3. **What does one row represent?** One unique customer order.
4. **Which column can be used as a classification target?** `is_late_delivery`.

In [4]:
# Step 4: Check Target Column
target = "is_late_delivery"

if target not in df.columns:
    raise ValueError(f"Target column {target} not found in the dataset.")

print("Target Value Counts:")
print(df[target].value_counts())

print("\nTarget Percentage Distribution:")
print(df[target].value_counts(normalize=True) * 100)

Target Value Counts:
is_late_delivery
0    92906
1     6535
Name: count, dtype: int64

Target Percentage Distribution:
is_late_delivery
0    93.428264
1     6.571736
Name: proportion, dtype: float64


### Student Checkpoint 2
1. **Is the target balanced or imbalanced?** Imbalanced (~93.4% on-time vs ~6.6% late).
2. **Why can accuracy be misleading if the target is imbalanced?** A dummy model predicting 0 for every order achieves ~93.4% accuracy while completely failing to catch late deliveries.
3. **Which metric should we also check?** Precision, Recall, and F1-score (especially Recall for class 1).

In [5]:
# Step 5: Define Columns to Remove
# Identifier columns cause memorization rather than learning
identifier_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

# Leakage columns contain post-delivery info or directly reveal the target
leakage_columns = [
    "delivery_days",
    "delivery_delay_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

columns_to_remove = identifier_columns + leakage_columns + [target]
columns_to_remove = [col for col in columns_to_remove if col in df.columns]

print("Columns to remove:", columns_to_remove)

Columns to remove: ['order_id', 'customer_id', 'customer_unique_id', 'delivery_days', 'delivery_delay_days', 'review_score', 'review_comment_count', 'has_review_comment', 'is_low_review', 'is_late_delivery']


In [6]:
# Step 6: Create Feature Matrix and Target Vector
X = df.drop(columns=columns_to_remove)
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (99441, 21)
y shape: (99441,)


### Student Checkpoint 3
1. **Why should `order_id` not be used as a model feature?** It is a high-cardinality unique tracking ID that leads to memorization rather than generalized learning.
2. **Why is `delivery_delay_days` unsafe for predicting `is_late_delivery`?** It directly calculates actual delivery date minus estimated delivery date; late delivery is defined as `delivery_delay_days > 0`, causing 100% target leakage.
3. **Why is `review_score` unsafe if prediction is made before delivery?** Reviews are submitted after delivery has taken place, making them unavailable at prediction time.

In [7]:
# Step 7: Identify Column Types Automatically
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['order_year', 'order_month', 'order_day', 'order_day_of_week', 'order_hour', 'estimated_delivery_days', 'total_payment_value', 'max_payment_installments', 'payment_types_count', 'total_items', 'total_price', 'total_freight', 'unique_products', 'unique_sellers']

Categorical features: ['customer_city', 'customer_state', 'order_status', 'dominant_payment_type', 'main_product_category', 'delivery_speed_category', 'payment_value_band']


In [8]:
# Step 8: Check Missing Values in Selected Features
print("Missing values in Numerical features:")
print(X[numeric_features].isnull().sum().sort_values(ascending=False).head(20))

print("\nMissing values in Categorical features:")
print(X[categorical_features].isnull().sum().sort_values(ascending=False).head(20))

Missing values in Numerical features:
total_items                 775
total_price                 775
total_freight               775
unique_products             775
unique_sellers              775
total_payment_value           1
max_payment_installments      1
payment_types_count           1
order_year                    0
order_month                   0
order_day                     0
order_day_of_week             0
order_hour                    0
estimated_delivery_days       0
dtype: int64

Missing values in Categorical features:
delivery_speed_category    2965
main_product_category      2185
dominant_payment_type         1
payment_value_band            1
customer_city                 0
customer_state                0
order_status                  0
dtype: int64


### Student Checkpoint 4
1. **Name three numerical features:** `total_payment_value`, `total_price`, `total_freight`.
2. **Name three categorical features:** `dominant_payment_type`, `main_product_category`, `customer_state`.
3. **Which type of feature needs one-hot encoding?** Categorical features.
4. **Which type of feature may need scaling?** Numerical features.

In [9]:
# Step 9: Split the Data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (79552, 21)
X_test shape: (19889, 21)
y_train shape: (79552,)
y_test shape: (19889,)


### Student Checkpoint 5
1. **Why did we use `stratify=y`?** To guarantee that both training and testing sets maintain the same ~6.6% ratio of late delivery orders as the original dataset.
2. **Why should preprocessing not be fitted before train-test split?** To prevent data leakage where parameters (mean, median, standard deviation, categories) from the test set contaminate the training process.

In [10]:
# Step 10 & 11: Create Numerical and Categorical Pipelines
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Step 12: Combine Pipelines with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### Student Checkpoint 6
1. **Why do numerical and categorical columns need different preprocessing?** Numerical columns require mathematical operations (imputation by median, standardization into z-scores), whereas categorical columns represent discrete qualitative labels requiring one-hot binary encoding.
2. **What is the role of `ColumnTransformer`?** It orchestrates disparate transformers across designated feature subsets and concatenates them into one combined array.
3. **Why is `handle_unknown='ignore'` useful?** It prevents pipeline failures when unseen categories occur in future test or production data.

In [11]:
# Step 13: Create Full Model Pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Step 14: Fit the Pipeline
print("Fitting model pipeline on training data...")
model_pipeline.fit(X_train, y_train)
print("Pipeline trained successfully!")

Fitting model pipeline on training data...
Pipeline trained successfully!


In [12]:
# Step 15 & 16: Predictions and Probabilities
y_pred = model_pipeline.predict(X_test)
y_pred_proba = model_pipeline.predict_proba(X_test)[:, 1]

print("Predicted probabilities for first 10 test orders:")
print(y_pred_proba[:10])

Predicted probabilities for first 10 test orders:
[2.25655410e-05 2.58063066e-03 8.65249856e-03 9.13191964e-05
 1.59420449e-05 3.34893883e-02 6.21160311e-05 8.86290294e-03
 1.85008652e-04 4.66953624e-03]


In [13]:
# Step 17: Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9373523052943838

Confusion Matrix:
[[17428  1154]
 [   92  1215]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.94      0.97     18582
           1       0.51      0.93      0.66      1307

    accuracy                           0.94     19889
   macro avg       0.75      0.93      0.81     19889
weighted avg       0.96      0.94      0.95     19889



### Student Checkpoint 7
1. **What does accuracy measure?** The fraction of total predictions that were correct: $(TP + TN) / \text{Total}$.
2. **Why can accuracy be misleading?** In imbalanced problems, high accuracy can conceal near-zero detection of critical minority outcomes.
3. **What does recall measure?** The fraction of actual positive (late) orders that the model identified: $TP / (TP + FN)$.
4. **Why may recall be important for late delivery prediction?** Catching late deliveries allows customer support to proactively alert customers, issue compensation, or expedite shipping before negative reviews occur.

In [14]:
# Step 18: Inspect Transformed Feature Names
feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()

print(f"Total number of transformed features: {len(feature_names)}")
print("\nFirst 20 feature names:")
print(feature_names[:20])

Total number of transformed features: 3981

First 20 feature names:
['num__order_year' 'num__order_month' 'num__order_day'
 'num__order_day_of_week' 'num__order_hour' 'num__estimated_delivery_days'
 'num__total_payment_value' 'num__max_payment_installments'
 'num__payment_types_count' 'num__total_items' 'num__total_price'
 'num__total_freight' 'num__unique_products' 'num__unique_sellers'
 'cat__customer_city_abadia dos dourados' 'cat__customer_city_abadiania'
 'cat__customer_city_abaete' 'cat__customer_city_abaetetuba'
 'cat__customer_city_abaiara' 'cat__customer_city_abaira']


In [15]:
# Step 19 & 20: Save and Reload Pipeline
output_model_path = Path("../models/late_delivery_pipeline.joblib")
output_model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model_pipeline, output_model_path)
print("Pipeline saved at:", output_model_path)

# Also save to root models/ for easy access
root_model_path = Path("models/late_delivery_pipeline.joblib")
root_model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model_pipeline, root_model_path)

# Reload and verify
loaded_pipeline = joblib.load(output_model_path)
loaded_predictions = loaded_pipeline.predict(X_test)
print("Loaded model predictions (first 10):", loaded_predictions[:10])

Pipeline saved at: ..\models\late_delivery_pipeline.joblib
Loaded model predictions (first 10): [0 0 0 0 0 0 0 0 0 0]


In [16]:
# Step 21: Predict on a Sample New Order
sample_order = X_test.iloc[[0]]
print("Sample Order Attributes:")
display(sample_order)

sample_pred = loaded_pipeline.predict(sample_order)[0]
sample_prob = loaded_pipeline.predict_proba(sample_order)[:, 1][0]

print(f"Prediction: {sample_pred} ({'Likely Late' if sample_pred == 1 else 'Likely On-Time'})")
print(f"Late Delivery Probability: {sample_prob:.4f}")

Sample Order Attributes:


,customer_city,customer_state,order_status,order_year,order_month,order_day,order_day_of_week,order_hour,estimated_delivery_days,total_payment_value,...,payment_types_count,dominant_payment_type,total_items,total_price,total_freight,unique_products,unique_sellers,main_product_category,delivery_speed_category,payment_value_band
33649,curitiba,PR,delivered,2018,1,4,3,13,25,154.66,...,1.0,credit_card,1.0,139.93,14.73,1.0,1.0,cool_stuff,Fast,High


Prediction: 0 (Likely On-Time)
Late Delivery Probability: 0.0000


In [17]:
# Part U: Extension Task - Replace with Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])

print("Training Random Forest Pipeline...")
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_pred))

Training Random Forest Pipeline...

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     18582
           1       0.92      0.49      0.64      1307

    accuracy                           0.96     19889
   macro avg       0.94      0.74      0.81     19889
weighted avg       0.96      0.96      0.96     19889



In [18]:
# Model Comparison Table
from sklearn.metrics import precision_score, recall_score, f1_score

rf_acc = accuracy_score(y_test, rf_pred)
lr_prec = precision_score(y_test, y_pred)
lr_rec = recall_score(y_test, y_pred)
lr_f1 = f1_score(y_test, y_pred)

rf_prec = precision_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [f"{accuracy:.4f}", f"{rf_acc:.4f}"],
    "Precision (Class 1)": [f"{lr_prec:.4f}", f"{rf_prec:.4f}"],
    "Recall (Class 1)": [f"{lr_rec:.4f}", f"{rf_rec:.4f}"],
    "F1-score (Class 1)": [f"{lr_f1:.4f}", f"{rf_f1:.4f}"]
})
display(comparison_df)

,Model,Accuracy,Precision (Class 1),Recall (Class 1),F1-score (Class 1)
0,Logistic Regression,0.9374,0.5129,0.9296,0.6610
1,Random Forest,0.9634,0.9165,0.4874,0.6364


### Part V: Discussion & Key Takeaways
1. **Which model performed better?**
   - **Logistic Regression (balanced)** achieved exceptional **Recall of 93%** on late deliveries, making it ideal if missing a late delivery is expensive.
   - **Random Forest (balanced)** achieved higher **Accuracy (96%)** and much higher **Precision of 92%** (minimizing false alarms), with 49% recall.
2. **Which model is easier to interpret?**
   - Logistic Regression is directly interpretable through its learned linear weights and feature odds ratios.
3. **Does Random Forest require scaling?**
   - Tree-based models are invariant to monotonic feature scaling, but keeping scaling in the preprocessor makes the pipeline interchangeable across algorithms.
4. **Why did we still keep preprocessing in the pipeline?**
   - It maintains strict reproducibility, eliminates data leakage, handles unseen categories, and enables seamless serialization and deployment (`joblib`).

---
**Preprocessing + Model + Evaluation + Reproducibility = Complete Machine Learning System**